In [1]:
import xarray as xr
import numpy as np
import plotly.graph_objects as go
from Montreal_UHI_toolbox import static_fields_C, add_field_to_stations, add_blurred_field_to_stations, obs, adjust_temp, Z_a
import scipy.stats as st
from UHI_statistics import UHI_seasonal, temps_seasonal

/runoff/gulley/.miniconda3/lib/python3.12/site-packages/gribapi/__init__.py:23: UserWarning: ecCodes 2.39.0 or higher is recommended. You are running version 2.14.1
  warnings.warn(


In [2]:
# Managing different elevations for temperature, temperature adjustments are performed in the final step of any rendering
# Based constant DABL assumption
# Extract blurred orography (effective model elevation) from simulation data at station points
adjustment_set = {}
adjustment_set = add_blurred_field_to_stations(static_fields_C['orog'],station_set = obs)

Z_b_model = adjustment_set['orog_blurred_std1p5'].values 
Z_b_model = Z_b_model + 2.*np.ones(len(Z_b_model))
print(f'orog (m) for each model station:\n{Z_b_model}\n...to be scaled to {Z_a}m\n\n')

Z_b_obs = adjustment_set['elev'].values
print(f'elev (m) for each actual station:\n{Z_b_obs}\n...to be scaled to {Z_a}m')

orog (m) for each model station:
[ 16.95971274 107.2258768   72.66397482  32.08885303  66.40041875
  57.84859445  34.24172871  45.84364934  26.54982949  33.92587983
  51.88732464  78.53580031]
...to be scaled to 54.5m


elev (m) for each actual station:
[21. 91. 68. 31. 61. 73. 36. 46. 27. 33. 49. 91.]
...to be scaled to 54.5m


In [ ]:
# Performing DABL temperature adjustments:
for season in ['JJA','SON','DJF','MAM']:
    for field in ['tasmin','tasmax','tas']:
        
        temps_seasonal[field]['C'][season]['vals'] = adjust_temp([[temps_seasonal[field]['C'][season]['vals'][i]] for i in range(len(temps_seasonal[field]['C'][season]['vals']))] ,z_b=Z_b_model).squeeze()
        temps_seasonal[field]['T'][season]['vals'] = adjust_temp([[temps_seasonal[field]['T'][season]['vals'][i]] for i in range(len(temps_seasonal[field]['T'][season]['vals']))] ,z_b=Z_b_model).squeeze()
        temps_seasonal[field]['S'][season]['vals'] = adjust_temp([[temps_seasonal[field]['S'][season]['vals'][i]] for i in range(len(temps_seasonal[field]['S'][season]['vals']))] ,z_b=Z_b_obs).squeeze()

In [16]:
for season in ['JJA','SON','DJF','MAM']:
    for field,field_name in zip(['tasmin','tasmax','tas'],
                    [f'<sup>{season}</sup><SPAN STYLE="text-decoration:overline">T</SPAN><sub>min</sub>',f'<sup>{season}</sup><SPAN STYLE="text-decoration:overline">T</SPAN><sub>max</sub>',f'<sup>{season}</sup><SPAN STYLE="text-decoration:overline">T</SPAN><sub>avg</sub>']):
    # for field,field_name in zip(['tasmin','tasmax'],
    #                 [f'<sup>{season}</sup><SPAN STYLE="text-decoration:overline">T</SPAN><sub>min</sub>','<SPAN STYLE="text-decoration:overline">T</SPAN><sub>max</sub>']):
        fig = go.Figure()

        # How much each model deviates from observation
        diff_C = temps_seasonal[field]['C'][season]['vals'] - temps_seasonal[field]['S'][season]['vals']
        diff_T = temps_seasonal[field]['T'][season]['vals'] - temps_seasonal[field]['S'][season]['vals']
        
        # Scatter station markers (observation vs observation)
        fig.add_trace(go.Scatter(
                x=temps_seasonal[field]['S'][season]['vals'],
                y=temps_seasonal[field]['S'][season]['vals'],
                mode='markers',
                hovertext=obs.station_name.values,
                hoverinfo='text+x+y',
                marker=dict(color='black'),
                hovertemplate=(
                'Station: %{customdata}<br>'
                'Observation: %{x:.2f}°C<br>'
                ),
                customdata=obs.station_name.values,
                legendgroup='Observation',
                legendgrouptitle={'text': 'Observation'},
                name='',
                error_y=dict(type='data', array=temps_seasonal[field]['S'][season]['ERR'], visible=True,color='rgba(0,0,0,0.2)'),
                meta=dict(has_error_bars=True)
            )
        )
        
        # Scatter CLASS-determined station temperatures (CLASS vs observation)
        fig.add_trace(go.Scatter(
                x=temps_seasonal[field]['S'][season]['vals'],
                y=temps_seasonal[field]['C'][season]['vals'],
                mode='markers',
                hovertext=obs.station_name.values,
                hoverinfo='text+x+y',
                marker=dict(color='blue'),
                hovertemplate=(
                'Station: %{customdata[0]}<br>'
                'Model (CLASS): %{y:.2f}°C<br>'
                'Observation: %{x:.2f}°C<br>'
                'ΔT: %{customdata[1]:+.2f}°C<extra></extra>'
                ),
                customdata=np.stack([obs.station_name.values, diff_C], axis=-1),
                legendgroup='CLASS',
                legendgrouptitle={'text': 'CLASS'},
                name='',
                error_y=dict(type='data', array=temps_seasonal[field]['C'][season]['ERR'], visible=True,color='rgba(0,0,255,0.2)'),
                meta=dict(has_error_bars=True)
            )
        )
        
        # Scatter TEB+CLASS-determined station temperatures (TEB+CLASS vs observation)
        fig.add_trace(go.Scatter(
                x=temps_seasonal[field]['S'][season]['vals'],
                y=temps_seasonal[field]['T'][season]['vals'],
                mode='markers',
                hovertext=obs.station_name.values,
                hoverinfo='text+x+y',
                marker=dict(color='red'),
                hovertemplate=(
                'Station: %{customdata[0]}<br>'
                'Model (TEB+CLASS): %{y:.2f}°C<br>'
                'Observation: %{x:.2f}°C<br>'
                'ΔT: %{customdata[1]:+.2f}°C<extra></extra>'
                ),
                customdata=np.stack([obs.station_name.values, diff_T], axis=-1),
                legendgroup='TEB+CLASS',
                legendgrouptitle={'text': 'TEB+CLASS'},
                name='',
                error_y=dict(type='data', array=temps_seasonal[field]['T'][season]['ERR'], visible=True,color='rgba(255,0,0,0.2)'),
                meta=dict(has_error_bars=True)
            )
        )
         

        # Error bar toggle
        error_traces = [i for i, tr in enumerate(fig.data)
                        if getattr(tr, "meta", {}).get("has_error_bars")]

        # Button callback
        buttons = [
            dict(
                label="Show error bars",
                method="restyle",
                args=[{"error_y.visible": True}, error_traces]
            ),
            dict(
                label="Hide error bars",
                method="restyle",
                args=[{"error_y.visible": False}, error_traces]
            )
        ]

        fig.update_layout(
            updatemenus=[dict(type="buttons", direction="right", buttons=buttons,
                            x=1.2, y=0, xanchor="center")]
        )
        
        
        # For setting the bounds of the graph 1:1 scale and aspect ratio
        max_xy = max([max(temps_seasonal[field]['C'][season]['vals']),max(temps_seasonal[field]['T'][season]['vals']),max(temps_seasonal[field]['S'][season]['vals'])]) + 1.0
        min_xy = min([max(temps_seasonal[field]['C'][season]['vals']),min(temps_seasonal[field]['T'][season]['vals']),min(temps_seasonal[field]['S'][season]['vals'])]) - 1.0

        # fig.update_xaxes(range=[min_xy,max_xy])
        # fig.update_yaxes(range=[min_xy,max_xy])
        # Or alternatively:
        midpoint = (min_xy + max_xy)/2
        window = 3
        min_xy = midpoint - window
        max_xy = midpoint + window
        fig.update_xaxes(range=[min_xy,max_xy])
        fig.update_yaxes(range=[min_xy,max_xy])
        

        # Draw observation line
        fig.add_trace(go.Scatter(
                x=[min_xy,max_xy],
                y=[min_xy,max_xy],
                mode='lines',
                line=dict(color='black', width=0.5),
                showlegend=False,
                hoverinfo='skip'
            ))

         # Draw vertical lines showing differences at each station
        for i in range(len(obs.station)):

            # Adding lines to show the differences from observation
            fig.add_trace(go.Scatter(
                x=[temps_seasonal[field]['S'][season]['vals'][i], temps_seasonal[field]['S'][season]['vals'][i]],
                y=[temps_seasonal[field]['S'][season]['vals'][i], temps_seasonal[field]['T'][season]['vals'][i]],
                mode='lines',
                line=dict(color='grey', width=0.2),
                showlegend=False,
                legendgroup='TEB+CLASS',
                legendgrouptitle={'text': 'TEB+CLASS'},
                hoverinfo='skip'
            ))
            fig.add_trace(go.Scatter(
                x=[temps_seasonal[field]['S'][season]['vals'][i], temps_seasonal[field]['S'][season]['vals'][i]],
                y=[temps_seasonal[field]['S'][season]['vals'][i], temps_seasonal[field]['C'][season]['vals'][i]],
                mode='lines',
                line=dict(color='grey', width=0.2),
                showlegend=False,
                legendgroup='CLASS',
                legendgrouptitle={'text': 'CLASS'},
                hoverinfo='skip'
            ))

        # Titling and layout
        fig.update_layout(
            xaxis_title=f'Observed {field_name} (°C)',
            yaxis_title=f'Modelled {field_name} (°C)',
            title=f'Modelled vs Observed {field_name}',
            width = 900,height=900
        )

        # fig.show()
        fig.write_html(f'/runoff/gulley/UHI_plots/accuracy/accuracy_{season}_{field}.html')